# Import Necessary Libraries

In [1]:
import pandas as pd 
import numpy as np 
from tqdm import tqdm 
import gzip as gz

# Wrangling Principals data

In [3]:
principals = pd.read_csv(gz.open("../datasets/title.principals.tsv.gz"),sep='\t',encoding='utf-8',lineterminator='\n')

In [4]:
principals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87845070 entries, 0 to 87845069
Data columns (total 6 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   tconst      object
 1   ordering    int64 
 2   nconst      object
 3   category    object
 4   job         object
 5   characters  object
dtypes: int64(1), object(5)
memory usage: 3.9+ GB


In [5]:
principals.drop('job',axis = 1, inplace=True)

In [6]:
principals.query(""" category == 'actor' | category == 'actress' | category == 'self' """)

,tconst,ordering,nconst,category,characters
0,tt0000001,1,nm1588970,self,"[""Self""]"
13,tt0000005,1,nm0443482,actor,"[""Blacksmith""]"
14,tt0000005,2,nm0653042,actor,"[""Assistant""]"
16,tt0000007,1,nm0179163,actor,\N
17,tt0000007,2,nm0183947,actor,\N
...,...,...,...,...,...
87845059,tt9916880,12,nm2676923,actress,"[""Sour Susan""]"
87845060,tt9916880,13,nm2676923,actress,"[""Goody-Goody Gordon""]"
87845061,tt9916880,14,nm2676923,actress,"[""Singing Soraya""]"
87845062,tt9916880,15,nm1469295,actress,"[""Perfect Peter""]"


In [8]:
principals.to_csv('../datasets/principals_test_data.csv',index='ignore')

# Filtering Data From null and "Self" values

In [2]:
target_principals = pd.read_csv('../datasets/sv_files/principals_test_data.csv')

In [3]:
target_principals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87845070 entries, 0 to 87845069
Data columns (total 6 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   Unnamed: 0  int64 
 1   tconst      object
 2   ordering    int64 
 3   nconst      object
 4   category    object
 5   characters  object
dtypes: int64(2), object(4)
memory usage: 3.9+ GB


In [4]:
target_principals.drop('Unnamed: 0',axis = 1, inplace=True)

In [6]:
target_principals.replace(r'\\N', "null" , regex=True)
target_principals.replace(r'/self/g', "null", regex=True)

,tconst,ordering,nconst,category,characters
0,tt0000001,1,nm1588970,self,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N
2,tt0000001,3,nm0005690,producer,\N
3,tt0000001,4,nm0374658,cinematographer,\N
4,tt0000002,1,nm0721526,director,\N
...,...,...,...,...,...
87845065,tt9916880,18,nm0996406,director,\N
87845066,tt9916880,19,nm1482639,writer,\N
87845067,tt9916880,20,nm2586970,writer,\N
87845068,tt9916880,21,nm1594058,producer,\N


In [7]:
movies = pd.read_csv('../datasets/sv_files/movies_df_with_collection_names.csv')[['id','imdb_id']]

In [8]:
tv_shows = pd.read_csv('../datasets/sv_files/tv_shows_df.csv')[['id','imdb_id']]

In [9]:
movies['type'] = 'movie'

In [10]:
tv_shows['type'] = 'tv show'

# Integrating with movies and tv shows data

In [11]:
target_items = pd.concat([movies,tv_shows],ignore_index=True)

In [12]:
df = pd.merge(target_items, target_principals, how='left', left_on='imdb_id',right_on='tconst').drop('tconst',axis = 1)

# Grouping Actor Information and Characters

In [13]:
grouped_data = df.groupby('imdb_id').agg({
    'ordering': list,
    'nconst': list,
    'category': list,
    'characters': list
}).reset_index()

In [14]:
df.to_csv('../datasets/imdb_characters_and_actor_ids.csv')